<a href="https://colab.research.google.com/github/KarthikSaravanakumarM56/24ADI006-24BAD056-DL/blob/main/DL_Lab_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Experiment No: 10
Part A: Environment Setup

In [1]:
# Roll No: 24BAD056
# Name: M. Karthik Saravanakumar

!pip install -q transformers torch gradio

Part A: Model Selection and Loading

In [2]:
# Roll No: 24BAD056
# Part A: Model Selection and Loading

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilgpt2"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Configure padding
tokenizer.pad_token = tokenizer.eos_token

# Select available device
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print("Model loaded successfully!")
print("Model Name:", MODEL_NAME)
print("Device:", device)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!
Model Name: distilgpt2
Device: cpu


In [3]:
# Roll No: 24BAD056
# Part A: Initial Prompt Testing

prompt = "Artificial Intelligence is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)
print("Prompt:", prompt)
print("Generated Response:", response)

Prompt: Artificial Intelligence is
Generated Response: Artificial Intelligence is a powerful technology that can transform us from a human to a machine. Artificial Intelligence is one of the most advanced technology in the world today.














PART B: PROMPT HANDLING AND RESPONSE GENERATION

In [4]:
# Roll No: 24BAD056
# Part B: Prompt Handling and Response Generation

def generate_response(user_prompt):
    try:
        # Check for empty input
        if not user_prompt or not user_prompt.strip():
            return "Please enter a valid prompt."

        # Convert text into tokens
        inputs = tokenizer(
            user_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=400
        ).to(device)

        # Generate response
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=80,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id
            )

        # Extract only newly generated tokens
        new_tokens = outputs[0][
            inputs["input_ids"].shape[1]:
        ]

        # Convert tokens into readable text
        response = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()
        if not response:
            return "The model could not generate a response."
        return response
    except Exception as e:
        return f"Error generating response: {e}"

# Test the function
prompt = "What is machine learning?"
print("User:", prompt)
print("Chatbot:", generate_response(prompt))

User: What is machine learning?
Chatbot: This book focuses on the process of recognizing and identifying patterns. The topic is presented by Dr. Michael Lachlan, a professor of psychology at Harvard University who leads the field of computer science at MIT and co-author of "Machine Learning: A Guide to Understanding Human Behavior." He also includes some valuable background information about how humans interact with machines in order to better understand what happens when they learn


In [5]:
# Roll No: 24BAD056
# Part B: Testing Multiple Prompts

test_prompts = [
    "What is artificial intelligence?",
    "Explain machine learning.",
    "What is Python programming?",
    "Tell me about neural networks.",
    "How does cloud computing work?"
]
for i, prompt in enumerate(test_prompts, start=1):
    print("\nTest", i)
    print("User:", prompt)
    response = generate_response(prompt)
    print("Chatbot:", response)


Test 1
User: What is artificial intelligence?
Chatbot: We need to understand what we are saying.
This post originally appeared on the New York Times and was republished with permission from The Conversation, Inc.

Test 2
User: Explain machine learning.
Chatbot: The model could not generate a response.

Test 3
User: What is Python programming?
Chatbot: I have a lot of problems with the language, but I am not going to get into it all completely. In fact, we are in quite some trouble and even more serious issues than before: It's like when you read about Ruby on Rails (and many others) that there might be no way for programmers to write code without having been trained by people who know what they're doing wrong or

Test 4
User: Tell me about neural networks.
Chatbot: I'm not talking about the brain, but my mind is telling you that we have to learn more and better ways to interact with other people in a way so as to be able to express ourselves differently within our lives."
The idea beh

PART C: BUILDING THE CHATBOT INTERFACE USING GRADIO

In [6]:
# Roll No: 24BAD056
# Part C: Building the Chatbot Interface

import gradio as gr
def chatbot(message, history):
    try:
        # Validate user input
        if not message or not message.strip():
            return "Please enter a message."

        # Prepare conversation context
        conversation = ""

        # Include the last three exchanges
        for item in history[-3:]:
            if isinstance(item, dict):

                # Handle message-based history
                role = item.get("role")
                content = item.get("content", "")
                if role == "user":
                    conversation += f"User: {content}\n"
                elif role == "assistant":
                    conversation += f"Assistant: {content}\n"
            elif isinstance(item, (list, tuple)) and len(item) == 2:

                # Handle older tuple-based history
                user_msg, bot_msg = item
                if user_msg:
                    conversation += f"User: {user_msg}\n"
                if bot_msg:
                    conversation += f"Assistant: {bot_msg}\n"

        # Add the current message
        conversation += f"User: {message}\nAssistant:"

        # Generate the response
        response = generate_response(conversation)
        return response
    except Exception as e:
        return f"An error occurred: {e}"

# Create the Gradio application
demo = gr.ChatInterface(
    fn=chatbot,
    title="Simple LLM Chatbot",
    description=(
        "A chatbot built using Python, "
        "Hugging Face Transformers, PyTorch, and Gradio."
    ),
    examples=[
        "What is artificial intelligence?",
        "Explain machine learning.",
        "What is Python programming?"
    ]
)

# Launch the application
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ba0914333e76f7c426.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


PART D: TESTING AND DEPLOYMENT

In [7]:
# Roll No: 24BAD056
# Part D: Testing the Chatbot

test_cases = [
    "What is artificial intelligence?",
    "Hello, how are you?",
    "Write a short story about a robot.",
    "Explain cloud computing.",
    "What is the use of Python?"
]
for i, prompt in enumerate(test_cases, start=1):
    print("\nTest Case:", i)
    print("Input:", prompt)
    response = generate_response(prompt)
    print("Output:", response)


Test Case: 1
Input: What is artificial intelligence?
Output: How do we make it possible to build a human brain that can perform tasks with real-time, non-intelligent computers (computer vision), and even machine learning algorithms for people without the need of data in order to solve problems like unemployment or health care? The answer lies in two ways: first, one of them is an Artificial Intelligence.
The second is a form of AI designed to be

Test Case: 2
Input: Hello, how are you?
Output: ”
A little bit of this is probably the first question I'll get into. What's your opinion on why people should take it seriously or not?

Test Case: 3
Input: Write a short story about a robot.
Output: The Robot Story: A Futuristic Book Series by John McDonough

Test Case: 4
Input: Explain cloud computing.
Output: com
If you have any questions, please email me at jimmy@joshkapelichelichelichec.org or visit www://www-joshkapelichelicheli.com

Test Case: 5
Input: What is the use of Python?
Output: W

In [9]:
# Roll No: 24BAD056

# Test 1: Empty Input Validation
print("TEST 1: Empty Input Validation")
prompt = ""
response = generate_response(prompt)

print("User:", prompt)
print("Chatbot:", response)

# Test 2: Conversational Input
print("\nTEST 2: Conversational Testing")

prompt = "Tell me about cloud computing."
response = generate_response(prompt)

print("User:", prompt)
print("Chatbot:", response)

# Test 3: Application Status
print("\nTEST 3: Application Demonstration")

print("LLM Chatbot Testing Completed.")
print("Model:", MODEL_NAME)
print("Interface: Gradio")
print("Environment: Google Colab")
print("Status: Ready for demonstration")

TEST 1: Empty Input Validation
User: 
Chatbot: Please enter a valid prompt.

TEST 2: Conversational Testing
User: Tell me about cloud computing.
Chatbot: I have no idea what it is like to work with a Linux system, so I am not trying to be too harsh on you guys in the comments or your thoughts here and there! If anything, please feel free to share my feedback below if that's any indication of how this works for some users:

TEST 3: Application Demonstration
LLM Chatbot Testing Completed.
Model: distilgpt2
Interface: Gradio
Environment: Google Colab
Status: Ready for demonstration
